# YOLO Classification Example

## YOLO Image Classification

This notebook is an example of image classification with **Ultralytics YOLO**.

```
Image (URL or file) → YOLO classifier → Top-N predicted labels + confidence scores
```

Unlike detection (bounding boxes for multiple objects), classification assigns **a single label** to the entire image, returning the top predicted classes ranked by confidence.

### Available models

| Model | Params | Speed |
|-------|--------|-------|
| `yolo11x-cls.pt` | 56.9 M | slowest / most accurate |
| `yolo11l-cls.pt` | 25.3 M | — |
| `yolo11m-cls.pt` | 20.1 M | — |
| `yolo11s-cls.pt` |  9.4 M | — |
| `yolo11n-cls.pt` |  2.6 M | fastest / lightest |

Change `MODEL_NAME` in the writefile cell to switch between them.

📄 [Ultralytics YOLO docs](https://docs.ultralytics.com/)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/CV-Samples/yolo"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls

In [ ]:
# Download sample images from GitHub
import os

# Image files to download
IMAGE_FILES = [
    'teefarm-pile-1651945_640.jpg',
]

BASE_URL = 'https://raw.githubusercontent.com/mastnk/cv-samples/main/yolo'
for fname in IMAGE_FILES:
    url = f'{BASE_URL}/{fname}'
    dest = f'{PROJECT_PATH}/{fname}'
    if not os.path.exists(dest):
        os.system(f'wget -q "{url}" -O "{dest}"')
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

%cd "{PROJECT_PATH}"
!ls


In [ ]:
# Install Ultralytics
!pip install ultralytics -q

import ultralytics
ultralytics.checks()  # check environment

## Using Your Own Images

There are two ways to provide images for classification:

**① Specify a URL**  
Pass a direct image URL to the `--url` flag when running the script:
```
%run classification.py --url https://example.com/image.jpg
```

**② Upload images to the `PROJECT_PATH/` folder**  
Place your image files in `PROJECT_PATH/`, then run the script with `--file` or `--dir`.

The easiest way to upload is through **Google Drive**:  
Open [drive.google.com](https://drive.google.com), navigate to `My Drive / CV-Samples / yolo/`, and drag & drop your files there.  
They will be immediately available in Colab via the mounted drive — no extra upload step needed.

```
%run classification.py --file your_image.jpg   # single file
%run classification.py --dir  .                # all images in folder
```

## Selecting a Model

In the writefile cell below, change `MODEL_NAME` to switch models.  
When multiple `MODEL_NAME = ...` lines are listed, **only the last one takes effect**.

```python
# MODEL_NAME = 'yolo11x-cls.pt'  # 56.9M params — most accurate, slowest
# MODEL_NAME = 'yolo11l-cls.pt'  # 25.3M params
# MODEL_NAME = 'yolo11m-cls.pt'  # 20.1M params
# MODEL_NAME = 'yolo11s-cls.pt'  #  9.4M params
MODEL_NAME   = 'yolo11n-cls.pt'  #  2.6M params — fastest, lightest  ← active
```

Larger models are more accurate but take longer to run. Start with `yolo11n-cls.pt` for quick experiments.

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile classification.py
"""YOLO Image Classification — command-line interface.

Usage:
  %run classification.py --url  <image_url>  [--disp] [--topk INT]
  %run classification.py --file <image_path> [--disp] [--topk INT]
  %run classification.py --dir  <image_dir>  [--disp] [--topk INT]
"""

import argparse
import glob
import os

from ultralytics import YOLO
from PIL import Image
import requests
from io import BytesIO

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
MODEL_NAME   = 'yolo11n-cls.pt'  #  2.6M params — fastest / lightest
# MODEL_NAME = 'yolo11s-cls.pt'  #  9.4M params
# MODEL_NAME = 'yolo11m-cls.pt'  # 20.1M params
# MODEL_NAME = 'yolo11l-cls.pt'  # 25.3M params
# MODEL_NAME = 'yolo11x-cls.pt'  # 56.9M params — most accurate

PROJECT_PATH = '/content/drive/MyDrive/CV-Samples/yolo'

# ---- Model loading -----------------------------------------------
model = YOLO(MODEL_NAME)

# ---- Display helper ----------------------------------------------
def show(image: Image.Image, label: str, disp: bool) -> None:
    """When --disp is active, display the image then print the filename/URL."""
    if disp:
        if _has_ipy:
            ipy_display(image)
        print(label)

# ---- Functions ---------------------------------------------------
def classify_url(url: str, topk: int = 5, disp: bool = False):
    """Download an image from a URL and classify it."""
    image   = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    results = model.predict(image, verbose=False)
    show(image, url, disp)
    probs = results[0].probs
    names = results[0].names
    top_indices = probs.top5[:topk]
    top_confs   = probs.top5conf[:topk].tolist()
    for idx, conf in zip(top_indices, top_confs):
        print(f'  {names[idx]:<30} {conf*100:.1f}%')
    return results


def classify_file(path: str, topk: int = 5, disp: bool = False):
    """Classify a single local image file."""
    image   = Image.open(path).convert('RGB')
    results = model.predict(image, verbose=False)
    show(image, path, disp)
    probs = results[0].probs
    names = results[0].names
    top_indices = probs.top5[:topk]
    top_confs   = probs.top5conf[:topk].tolist()
    for idx, conf in zip(top_indices, top_confs):
        print(f'  {names[idx]:<30} {conf*100:.1f}%')
    return results


def classify_dir(directory: str, topk: int = 5, disp: bool = False):
    """Classify all images in a directory."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    for path in filepaths:
        classify_file(path, topk, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='YOLO Image Classification')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--url',  type=str, help='Image URL to classify')
group.add_argument('--file', type=str, help='Path to a single image file')
group.add_argument('--dir',  type=str, help='Directory of images to classify')
parser.add_argument('--topk', type=int, default=5,
                    help='Number of top predictions to show (default: 5)')
parser.add_argument('--disp', action='store_true',
                    help='Display image and filename/URL before results')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.url:
    classify_url(args.url,   topk=args.topk, disp=args.disp)
elif args.file:
    classify_file(args.file, topk=args.topk, disp=args.disp)
elif args.dir:
    classify_dir(args.dir,   topk=args.topk, disp=args.disp)


## `classification.py` Usage

After running the `%%writefile` cell above, `classification.py` is saved in `PROJECT_PATH`.
Run it with **`%run`** (not `!python`) so that inline image display works in Colab.

```
%run classification.py --url  <image_url>        # classify a remote image
%run classification.py --file <image_path>       # classify a single local file
%run classification.py --dir  <image_directory>  # classify all images in a folder
```

**Optional arguments**

| Flag | Default | Description |
|------|---------|-------------|
| `--disp` | off | Display image and filename / URL before results |
| `--topk <n>` | `5` | Number of top predictions to show (1 – 5) |

**Examples**

```bash
# Classify a remote image and display it
%run classification.py --url https://cdn.pixabay.com/photo/2016/12/13/05/15/puppy-1903313_640.jpg --disp

# Classify one file, show only top-1 prediction
%run classification.py --file teefarm-pile-1651945_640.jpg --topk 1 --disp

# Classify every image in the folder, display each one
%run classification.py --dir . --disp
```

**Output format (with `--disp`)**

```
<image displayed inline>
teefarm-pile-1651945_640.jpg
  golden_retriever               82.4%
  Labrador_retriever             10.1%
  tennis_ball                     3.2%
  ...
```

## Execution Methods

Use **`%run`** to execute `classification.py` inside the Colab kernel — this enables inline image display with `--disp`.

| Mode | Flag | Description |
|------|------|-------------|
| From URL | `--url <url>` | Fetches and classifies a remote image |
| Single file | `--file <path>` | Classifies one local image |
| Directory | `--dir <path>` | Classifies all images in a folder |

Add `--disp` to display each image and its filename / URL before the results.  
Add `--topk <n>` to change the number of top predictions shown (default: 5).

In [ ]:
# Execute: classify an image from URL (with display)
%cd "{PROJECT_PATH}"
%run classification.py --disp --url https://cdn.pixabay.com/photo/2016/12/13/05/15/puppy-1903313_640.jpg


In [ ]:
# Execute: classify a single local image file (with display)
%cd "{PROJECT_PATH}"
%run classification.py --disp --file teefarm-pile-1651945_640.jpg


In [ ]:
# Execute: classify all images in a directory (with display)
%cd "{PROJECT_PATH}"
%run classification.py --disp --dir .


💾 **Don't forget to save this notebook!**

To keep your work, go to **File → Save a copy in Drive** before closing.

